# Generate the AE dimension and architecture ablations.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score, confusion_matrix)


DATA_DIR = Path("data")
PROTOCOL_DIR = Path("revision_outputs") / "02_FD001_protocol"
OUTPUT_ROOT = Path("revision_outputs") / "06_FD001_AE_ablation"
FOLDS = tuple(range(1,11))
SEEDS = (42,43,44)
SENSORS = [f"Sensor{i}" for i in range(1,22)]
COLUMNS = ["UnitNumber", "Time", "OpSet1", "OpSet2", "OpSet3"] + SENSORS
EXPECTED_PROTOCOL = "880c712efeb9fdbdeec4d940c24247f7b1bab5c69a00e098106f88fc9c6774e8"
CANDIDATES = {"IF": [{"n_estimators":n} for n in [100,200,400]],
              "LOF": [{"n_neighbors":n} for n in [20,50,100]],
              "kNN": [{"n_neighbors":n} for n in [20,50,100]]}

CONFIGS = [
    {"name":"AE_deep_d4", "architecture":"deep", "latent_dim":4},
    {"name":"AE_deep_d8", "architecture":"deep", "latent_dim":8},
    {"name":"AE_shallow_d4", "architecture":"shallow", "latent_dim":4},
    {"name":"AE_shallow_d8", "architecture":"shallow", "latent_dim":8},
    {"name":"AE_shallow_d21", "architecture":"shallow", "latent_dim":21},
]
IMPLEMENTATION_ID = "FD001_AE_ablation_v1"


In [ ]:
try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError("Bu notebook TensorFlow gerektiriyor. TensorFlow kurulu kernel'i seçin; yoksa bu hata mesajını paylaşın.") from exc
print("TensorFlow:",tf.__version__)
print("Python:",platform.python_version(),"| scikit-learn:",sklearn.__version__)


In [ ]:
def require(ok, message):
    if not bool(ok): raise ValueError(message)

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def choose_threshold(y, scores):
    require(np.isfinite(scores).all() and len(np.unique(y))==2, "Geçersiz seçim skorları/etiketleri.")
    p,r,t=precision_recall_curve(y, scores)
    f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(t),where=(p[:-1]+r[:-1])>0)

    i=np.flatnonzero(np.isclose(f,f.max(),rtol=0,atol=1e-12))[-1]
    return float(t[i])

def fit_candidate(method, params, X, seed):
    if method=="IF":
        model=IsolationForest(**params,max_samples="auto",contamination="auto",random_state=seed,n_jobs=-1)
    elif method=="LOF":
        require(params["n_neighbors"]<len(X), "LOF için yetersiz fit örneği.")
        model=LocalOutlierFactor(**params,novelty=True,contamination="auto",metric="euclidean",n_jobs=-1)
    else:
        require(params["n_neighbors"]<=len(X), "kNN için yetersiz fit örneği.")
        model=NearestNeighbors(**params,metric="euclidean",n_jobs=-1)
    return model.fit(X)

def score(method, model, X):


    values = (model.kneighbors(X,return_distance=True)[0][:,-1]
              if method=="kNN" else -model.score_samples(X))
    require(np.isfinite(values).all(), "Sonlu olmayan model skoru.")
    return values

def metrics(y,scores,threshold):
    pred=(scores>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"AP":float(average_precision_score(y,scores)),
            "ROC_AUC":float(roc_auc_score(y,scores)),
            "precision":float(precision_score(y,pred,zero_division=0)),
            "recall":float(recall_score(y,pred,zero_division=0)),
            "F1":float(f1_score(y,pred,zero_division=0)),
            "Macro_F1":float(f1_score(y,pred,average="macro",zero_division=0)),
            "TP":int(tp),"FP":int(fp),"TN":int(tn),"FN":int(fn)}


In [ ]:
BASELINE_SHA256 = '5256e46907b1bf708c00d6057df45ec513f2b8658ea85f105511031551caad12'
BASELINE = load_json('assets/06/05_baseline.json')


In [ ]:
summary_path=PROTOCOL_DIR/"protocol_summary.json"
require(summary_path.is_file(), f"Dosya yok: {summary_path}")
stage02=json.loads(summary_path.read_text(encoding="utf-8"))
require(stage02["status"]=="PASS" and stage02["canonical_protocol_sha256"]==EXPECTED_PROTOCOL,
        "Kontrol edilen 02 protokolüyle uyuşmuyor.")
for name, expected in stage02["output_file_sha256"].items():
    require((PROTOCOL_DIR/name).is_file() and digest(PROTOCOL_DIR/name)==expected,
            f"02 çıktısı eksik/değişmiş: {name}")
protocol=json.loads((PROTOCOL_DIR/"experiment_protocol.json").read_text(encoding="utf-8"))
canonical=json.dumps(protocol,sort_keys=True,ensure_ascii=False,separators=(",",":"))
require(hashlib.sha256(canonical.encode()).hexdigest()==EXPECTED_PROTOCOL,"Protokol içeriği farklı.")
train_path=DATA_DIR/"train_FD001.txt"
require(train_path.is_file(),f"Dosya yok: {train_path}")
require(digest(train_path)==protocol["source_raw_file_sha256_from_stage01"]["train_FD001.txt"],
        "Ham eğitim dosyası 01 aşamasındaki dosyadan farklı.")
raw=pd.read_csv(train_path,sep=r"\s+",header=None)
require(raw.shape==(20631,26),"Ham veri boyutu farklı.")
raw.columns=COLUMNS
require(np.isfinite(raw.to_numpy(dtype=float)).all(),"Ham veride eksik/sonsuz değer.")
reference=pd.read_csv(PROTOCOL_DIR/"train_evaluation_reference.csv")
require(len(reference)==len(raw) and np.array_equal(reference.source_row,np.arange(len(raw))),"Metadata sırası farklı.")
require(np.array_equal(raw[["UnitNumber","Time"]].to_numpy(),reference[["UnitNumber","Time"]].to_numpy()),
        "Sensör ve metadata satırları eşleşmiyor.")
require(np.array_equal(reference.late_life_le_30,reference.RUL.le(30).astype(int)),"Referans etiketi farklı.")
splits=pd.read_csv(PROTOCOL_DIR/"engine_splits.csv")

require(FOLDS == tuple(range(1,11)) and SEEDS == (42,43,44),"Bu karşılaştırmada fold/seed ayarlarını değiştirmeyin.")
require(not splits.duplicated(["scheme","fold","engine_key"]).any(),"Tek deneyde motor rolü tekrarlanmış.")
outer=splits.loc[splits.scheme.eq("cv") & splits.role.eq("outer_valid")]
require(outer.engine_key.value_counts().eq(1).all() and outer.engine_key.nunique()==100,
        "Her motor dış değerlendirmeye tam bir kez girmeli.")

require(BASELINE["status"]=="PASS" and BASELINE["completed_runs"]==30,
        "Önceki CV özeti tamamlanmamış.")
require(BASELINE["protocol_sha256"]==EXPECTED_PROTOCOL and not BASELINE["test_files_read"],
        "Önceki CV protokolü farklı.")
versions={"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,
          "sklearn":sklearn.__version__,"tensorflow":tf.__version__}
for key,value in BASELINE["versions"].items():
    require(versions[key]==value,
            f"Ortam 05 ile farklı: {key}={versions[key]}, beklenen={value}. Python (C-MAPSS Revision) kernel'ini seçin.")
for window in [20,30,40]:
    require(np.array_equal(reference[f"late_life_le_{window}"],reference.RUL.le(window).astype(int)),
            f"RUL {window} referansı farklı.")

run_plan={"stage":"06_AE_ablation", "implementation_id":IMPLEMENTATION_ID,
          "protocol_sha256":EXPECTED_PROTOCOL,"baseline_file_sha256":BASELINE_SHA256,
          "raw_train_sha256":digest(train_path),"versions":versions,"keras_version":tf.keras.__version__,
          "folds":list(FOLDS),"seeds":list(SEEDS),"new_configurations":CONFIGS,
          "reused_configuration":"AE_deep_d21 (05 AE_latent21)","candidates":CANDIDATES,
          "training":{"max_epochs":100,"patience":10,"batch_size":256,"Adam_lr":0.001,
                      "loss":"mse","restore_best_weights":True,"shuffle":True},
          "selection":"per configuration: selection AP, then selection positive F1 threshold",
          "primary_reference":"RUL<=30; sensitivity 20 and 40 with unchanged scores/threshold",
          "inference":"descriptive follow-up after viewing stage05; not architecture selection validation",
          "test_files_read":False}
plan_text=json.dumps(run_plan,sort_keys=True,ensure_ascii=False,separators=(",",":"))
PLAN_SHA256=hashlib.sha256(plan_text.encode()).hexdigest()
RUN_DIR=OUTPUT_ROOT/PLAN_SHA256[:20]
RUN_DIR.mkdir(parents=True,exist_ok=True)
plan_file=RUN_DIR/"run_plan.json"
if plan_file.is_file():
    require(json.loads(plan_file.read_text(encoding="utf-8"))==run_plan,"Kayıtlı çalışma planı farklı.")
else:
    plan_file.write_text(json.dumps(run_plan,ensure_ascii=False,indent=2),encoding="utf-8")
(RUN_DIR/"stage05_baseline_summary.json").write_text(json.dumps(BASELINE,indent=2),encoding="utf-8")
print("Girdi ve ortam kontrolü tamamlandı. Çıktı klasörü:",RUN_DIR.resolve())


In [ ]:
def prepare_fold(fold_id):
    fold=splits.loc[splits.scheme.eq("cv") & splits.fold.eq(fold_id)].copy()
    require(fold.engine_key.is_unique and len(fold)==100,"Motor rol çakışması/eksik motor.")
    require(fold.role.value_counts().to_dict()=={"fit":70,"early_stop":10,"selection":10,"outer_valid":10},
            "Beklenen motor rolleri farklı.")
    roles=reference.engine_key.map(fold.set_index("engine_key").role)
    require(roles.notna().all(),"Atanmamış motor var.")
    masks={role:roles.eq(role).to_numpy() for role in ["fit","early_stop","selection","outer_valid"]}
    X=raw[SENSORS].to_numpy(dtype=np.float32)
    scaler=StandardScaler().fit(X[masks["fit"]])
    Xs={role:scaler.transform(X[mask]) for role,mask in masks.items()}
    y_selection=reference.loc[masks["selection"],"late_life_le_30"].to_numpy(dtype=int)
    require(len(np.unique(y_selection))==2,"Seçim kümesi tek sınıf.")
    return masks,scaler,Xs,y_selection

def build_ae(config,seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    d=config["latent_dim"]
    inputs=tf.keras.Input(shape=(21,),name="sensor_input")
    h=tf.keras.layers.Dense(42,activation="relu",name="encoder_42")(inputs)
    if config["architecture"]=="deep":
        h=tf.keras.layers.Dense(21,activation="relu",name="encoder_21")(h)
    latent=tf.keras.layers.Dense(d,activation="linear",name="latent")(h)
    h=latent
    if config["architecture"]=="deep":
        h=tf.keras.layers.Dense(21,activation="relu",name="decoder_21")(h)
    h=tf.keras.layers.Dense(42,activation="relu",name="decoder_42")(h)
    outputs=tf.keras.layers.Dense(21,activation="linear",name="reconstruction")(h)
    ae=tf.keras.Model(inputs,outputs,name=config["name"])
    encoder=tf.keras.Model(inputs,latent,name=config["name"]+"_encoder")
    ae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),loss="mse")
    return ae,encoder

def fit_representation(config,seed,Xs,OUT):
    ae,encoder=build_ae(config,seed)
    early=tf.keras.callbacks.EarlyStopping(monitor="val_loss",mode="min",patience=10,restore_best_weights=True)
    started=time.perf_counter()
    history=ae.fit(Xs["fit"],Xs["fit"],validation_data=(Xs["early_stop"],Xs["early_stop"]),
                   epochs=100,batch_size=256,shuffle=True,callbacks=[early],verbose=0)
    elapsed=time.perf_counter()-started
    require(np.isfinite(history.history["loss"]).all() and np.isfinite(history.history["val_loss"]).all(),
            "AE eğitiminde sonlu olmayan loss.")
    ae.save(OUT/"autoencoder.keras")
    encoder.save(OUT/"encoder.keras")
    pd.DataFrame(history.history).assign(epoch=np.arange(1,len(history.history["loss"])+1)).to_csv(OUT/"ae_history.csv",index=False)
    Z={role:encoder.predict(Xs[role],batch_size=256,verbose=0) for role in ["fit","selection","outer_valid"]}
    for role,z in Z.items():
        require(z.shape==(len(Xs[role]),config["latent_dim"]) and np.isfinite(z).all(),"Geçersiz latent çıktı.")
    latent_scaler=StandardScaler().fit(Z["fit"])
    Zs={role:latent_scaler.transform(z) for role,z in Z.items()}
    ae_scores={}
    for role in ["selection","outer_valid"]:
        reconstructed=ae.predict(Xs[role],batch_size=256,verbose=0)
        ae_scores[role]=np.mean(np.square(Xs[role]-reconstructed),axis=1)
        require(np.isfinite(ae_scores[role]).all(),"Geçersiz reconstruction skoru.")
    ae_info={"epochs_run":len(history.history["loss"]),
             "best_epoch":int(np.argmin(history.history["val_loss"])+1),
             "best_early_stop_loss":float(min(history.history["val_loss"])),
             "final_training_loss":float(history.history["loss"][-1]),
             "final_early_stop_loss":float(history.history["val_loss"][-1]),
             "parameter_count":int(ae.count_params()),"training_seconds":elapsed,
             "latent_fit_std":Z["fit"].std(axis=0).tolist()}
    del ae,encoder
    return Zs,ae_scores,latent_scaler,ae_info

def run_job(config,fold_id,seed):
    OUT=RUN_DIR/config["name"]/f"seed_{seed}"/f"fold_{fold_id:02d}"
    marker=OUT/"result.json"
    if marker.is_file():
        cached=json.loads(marker.read_text(encoding="utf-8"))
        require(cached["plan_sha256"]==PLAN_SHA256 and cached["config"]==config
                and cached["fold"]==fold_id and cached["seed"]==seed,"Kayıtlı iş kimliği farklı.")
        require(cached["status"]=="PASS" and len(cached["metrics"])==12,"Kayıtlı iş eksik.")
        for name,expected in cached["output_sha256"].items():
            require((OUT/name).is_file() and digest(OUT/name)==expected,f"Tamamlanmış çıktı değişmiş: {OUT/name}")
        print("KAYITTAN:",config["name"],"seed",seed,"fold",fold_id,flush=True)
        return cached
    OUT.mkdir(parents=True,exist_ok=True)
    masks,scaler,Xs,y_selection=prepare_fold(fold_id)
    Zs,ae_scores,latent_scaler,ae_info=fit_representation(config,seed,Xs,OUT)
    chosen={}
    trials=[]
    for method,candidates in CANDIDATES.items():
        best=None
        for i,params in enumerate(candidates):
            model=fit_candidate(method,params,Zs["fit"],seed)
            selection_scores=score(method,model,Zs["selection"])
            ap=float(average_precision_score(y_selection,selection_scores))
            trials.append({"method":method,"candidate_order":i,"params":params,"selection_AP":ap})
            if best is None or ap>best["selection_AP"]:
                best={"model":model,"params":params,"selection_AP":ap,"scores":selection_scores}
        best["threshold"]=choose_threshold(y_selection,best["scores"])
        chosen[method]=best
    chosen["MSE"]={"params":{},"selection_AP":float(average_precision_score(y_selection,ae_scores["selection"])),
                   "threshold":choose_threshold(y_selection,ae_scores["selection"])}

    held=reference.loc[masks["outer_valid"]].reset_index(drop=True)
    rows=[]
    frames=[]
    for method,best in chosen.items():
        scores=ae_scores["outer_valid"] if method=="MSE" else score(method,best["model"],Zs["outer_valid"])
        for window in [20,30,40]:
            y=held[f"late_life_le_{window}"].to_numpy(dtype=int)
            require(len(np.unique(y))==2,"Dış referans tek sınıf.")
            rows.append({"representation":config["name"],"architecture":config["architecture"],
                         "latent_dim":config["latent_dim"],"seed":seed,"fold":fold_id,"method":method,
                         "RUL_window":window,"selected_params":best["params"],
                         "selection_AP_primary":best["selection_AP"],"threshold":best["threshold"],
                         **metrics(y,scores,best["threshold"])})
        frame=held[["sample_key","engine_key","UnitNumber","Time","RUL",
                    "late_life_le_20","late_life_le_30","late_life_le_40"]].copy()
        frame["method"]=method
        frame["score"]=scores
        frame["threshold"]=best["threshold"]
        frame["prediction"]=(scores>=best["threshold"]).astype(int)
        frames.append(frame)
    pd.concat(frames,ignore_index=True).to_csv(OUT/"outer_predictions.csv",index=False)
    pd.DataFrame(rows).to_csv(OUT/"metrics.csv",index=False)
    (OUT/"selection_trials.json").write_text(json.dumps(trials,indent=2),encoding="utf-8")
    scaler_info={"input":{"features":SENSORS,"mean":scaler.mean_.tolist(),"scale":scaler.scale_.tolist()},
                 "latent":{"mean":latent_scaler.mean_.tolist(),"scale":latent_scaler.scale_.tolist()},
                 "fit_rows":int(masks["fit"].sum())}
    (OUT/"scaler_parameters.json").write_text(json.dumps(scaler_info,indent=2),encoding="utf-8")
    result={"status":"PASS","plan_sha256":PLAN_SHA256,"config":config,"fold":fold_id,"seed":seed,
            "AE":ae_info,"metrics":rows,"selection_trials":trials,
            "roles":{role:{"engines":int(reference.loc[mask,"engine_key"].nunique()),"rows":int(mask.sum())}
                     for role,mask in masks.items()},"test_files_read":False,
            "output_sha256":{p.name:digest(p) for p in OUT.iterdir() if p.is_file() and p.name not in ["result.json","result.tmp"]}}
    temp=OUT/"result.tmp"
    temp.write_text(json.dumps(result,indent=2),encoding="utf-8")
    temp.replace(marker)
    print("BİTTİ:",config["name"],"seed",seed,"fold",fold_id,
          "epoch",ae_info["epochs_run"],"best",ae_info["best_epoch"],flush=True)
    return result


In [ ]:
jobs=[]
total=len(CONFIGS)*len(SEEDS)*len(FOLDS)
for config in CONFIGS:
    for seed in SEEDS:
        for fold_id in FOLDS:
            print(f"[{len(jobs)+1}/{total}] {config['name']} — seed={seed}, fold={fold_id}",flush=True)
            jobs.append(run_job(config,fold_id,seed))
print("150 ek AE işi tamamlandı.")


In [ ]:
METRICS=["AP","ROC_AUC","precision","recall","F1","Macro_F1"]

def normalize_baseline(row):
    r=dict(row)
    r["representation"]="Raw" if row["representation"]=="Raw" else "AE_deep_d21"
    return r

def check_coverage(table,expected_groups):
    keys=["seed","fold","representation","method","RUL_window"]
    require(not table.duplicated(keys).any(),"Tekrarlanan deney sonucu.")
    groups=list(table.groupby(["representation","method","RUL_window"]))
    require(len(groups)==expected_groups,"Beklenen deney grupları oluşmadı.")
    wanted={(s,f) for s in SEEDS for f in FOLDS}
    for key,g in groups:
        require(len(g)==30 and set(zip(g.seed,g.fold))==wanted,f"Eksik fold/seed: {key}")

def summarize_ablation(jobs):
    require(len(jobs)==150,"150 ek eğitim tamamlanmadı.")
    require({(x["config"]["name"],x["seed"],x["fold"]) for x in jobs}
            =={(c["name"],s,f) for c in CONFIGS for s in SEEDS for f in FOLDS},"İş kapsamı farklı.")
    new=pd.DataFrame([r for job in jobs for r in job["metrics"]])
    require(len(new)==1800,"Beklenen 150 × 4 × 3 metrik satırı yok.")
    check_coverage(new,60)
    baseline_primary=pd.DataFrame([normalize_baseline(r) for r in BASELINE["fold_primary_results"]])
    primary=pd.concat([baseline_primary,new.loc[new.RUL_window.eq(30)]],ignore_index=True)
    check_coverage(primary,27)
    overall=[normalize_baseline(r) for r in BASELINE["primary_results"]+BASELINE["sensitivity_results"]]
    per_seed=[normalize_baseline(r) for r in BASELINE["per_seed_results"]]
    for (rep,method,window),g in new.groupby(["representation","method","RUL_window"],sort=True):
        row={"representation":rep,"method":method,"RUL_window":int(window),"runs":30}
        for metric in METRICS:
            row[metric+"_mean"]=float(g[metric].mean())
            row[metric+"_descriptive_sd"]=float(g[metric].std(ddof=1))
        overall.append(row)
        for seed,gs in g.groupby("seed"):
            item={"seed":int(seed),"representation":rep,"method":method,"RUL_window":int(window)}
            for metric in METRICS:
                item[metric+"_fold_mean"]=float(gs[metric].mean())
                item[metric+"_fold_sd"]=float(gs[metric].std(ddof=1))
            per_seed.append(item)

    contrasts=[]
    for config_name in ["AE_deep_d21"]+[c["name"] for c in CONFIGS]:
        for method in CANDIDATES:
            contrasts.append(("representation_vs_raw",config_name,"Raw",method))
    for d in [4,8,21]:
        for method in [*CANDIDATES,"MSE"]:
            contrasts.append(("architecture_deep_minus_shallow",f"AE_deep_d{d}",f"AE_shallow_d{d}",method))
    for arch in ["deep","shallow"]:
        for d in [4,8]:
            for method in [*CANDIDATES,"MSE"]:
                contrasts.append(("dimension_smaller_minus_21",f"AE_{arch}_d{d}",f"AE_{arch}_d21",method))
    paired=[]
    for kind,left,right,method in contrasts:
        a=primary.loc[primary.representation.eq(left)&primary.method.eq(method)].set_index(["seed","fold"])
        b=primary.loc[primary.representation.eq(right)&primary.method.eq(method)].set_index(["seed","fold"])
        require(len(a)==30 and len(b)==30 and set(a.index)==set(b.index),"Eşleştirme eksik.")
        row={"contrast":kind,"left":left,"right":right,"method":method,"pairs":30}
        for metric in ["AP","F1"]:
            delta=a[metric]-b[metric]
            row["delta_"+metric+"_mean"]=float(delta.mean())
            row["delta_"+metric+"_descriptive_sd"]=float(delta.std(ddof=1))
            row["positive_"+metric+"_runs"]=int((delta>1e-12).sum())
            row["negative_"+metric+"_runs"]=int((delta< -1e-12).sum())
            row["tied_"+metric+"_runs"]=int((delta.abs()<=1e-12).sum())
        paired.append(row)
    training=[dict(representation="AE_deep_d21",**x) for x in BASELINE["AE_training"]]
    training += [dict(representation=x["config"]["name"],seed=x["seed"],fold=x["fold"],**x["AE"]) for x in jobs]
    new.to_csv(RUN_DIR/"new_fold_metrics.csv",index=False)
    primary.to_csv(RUN_DIR/"combined_primary_fold_metrics.csv",index=False)
    pd.DataFrame(overall).to_csv(RUN_DIR/"aggregate_metrics.csv",index=False)
    pd.DataFrame(per_seed).to_csv(RUN_DIR/"per_seed_metrics.csv",index=False)
    pd.DataFrame(paired).to_csv(RUN_DIR/"paired_contrasts.csv",index=False)
    summary={"stage":"06_AE_ablation","status":"PASS","completed_new_AE_runs":len(jobs),
             "reused_AE_runs":30,"plan_sha256":PLAN_SHA256,"protocol_sha256":EXPECTED_PROTOCOL,
             "baseline_file_sha256":BASELINE_SHA256,"versions":versions,
             "folds":list(FOLDS),"seeds":list(SEEDS),"new_configurations":CONFIGS,
             "test_files_read":False,"statistical_tests_performed":False,
             "reference":"model-output-independent RUL late-life proxy, not annotated anomaly onset",
             "primary_RUL_window":30,"training_budget_epochs":100,
             "interpretation":"descriptive follow-up ablation; no winner selected on outer folds; SD is not CI",
             "primary_results":[r for r in overall if r["RUL_window"]==30],
             "sensitivity_results":[r for r in overall if r["RUL_window"]!=30],
             "per_seed_results":per_seed,"paired_primary_contrasts":paired,
             "AE_training":training,"fold_primary_results":primary.to_dict("records")}

    summary["fold_primary_results"]=[{k:v for k,v in r.items() if not (isinstance(v,float) and np.isnan(v))}
                                     for r in summary["fold_primary_results"]]
    output=RUN_DIR/"ablation_summary.json"
    output.write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
    return summary

ablation_summary=summarize_ablation(jobs)
print(pd.DataFrame(ablation_summary["primary_results"])[["representation","method","AP_mean","F1_mean"]].to_string(index=False))
print("ABLATION: PASS — bütün işler tamamlandı; üstünlük veya yakınsama iddiası değildir.")
print("Gönderilecek dosya:",(RUN_DIR/"ablation_summary.json").resolve())
